# Planning a multidimensional sweep

PyNST builds sweep plans as ordered pandas `MultiIndex` objects. This notebook combines independent ranges with a correlated complex point pattern, validates the result, and estimates its duration.

In [ ]:
from datetime import datetime

from pynst.planning import (
    PointPattern,
    combine_multiindexes,
    create_multiindex,
    create_range,
    estimate_sweep_time,
    validate_grid,
)

## Independent scalar axes

`create_range` retains the requested bounds. If a requested step does not divide the span exactly, PyNST adjusts it explicitly and emits a warning instead of silently overshooting.

In [ ]:
bias = create_range(min=0.0, max=1.0, step=0.25)
frequency = create_range(min=1.0e9, max=2.0e9, num=3)
scalar_grid = create_multiindex(
    [bias, frequency],
    names=["bias_V", "frequency_Hz"],
)
scalar_grid

## Correlated complex points

Magnitude and angle describe one physical point and therefore stay correlated. `mode="existing"` preserves the tuples already present in the point-pattern index.

In [ ]:
pattern = PointPattern()
pattern.create_circ(radius=0.5, step=0.25, include_center=True)
gamma = pattern.get_mi(
    format="ma",
    names=("gamma_magnitude", "gamma_angle_deg"),
)
sweep_grid = combine_multiindexes(scalar_grid, gamma, mode="existing")
sweep_grid[:8]

## Validation and duration

A correlated pattern is intentionally not the full Cartesian product of magnitude and angle. We validate the scalar grid as regular and keep the point-pattern tuples intact in the combined plan.

In [ ]:
report = validate_grid(scalar_grid)
estimate = estimate_sweep_time(
    len(sweep_grid),
    seconds_per_point=0.2,
    now=datetime(2026, 1, 1, 12, 0),
)
print(report.summary())
print(f"planned points: {len(sweep_grid)}")
print(f"duration: {estimate.total_duration}")
print(f"estimated finish: {estimate.estimated_finish}")